# 02 — Exploratory Data Analysis

Inspects the structure, completeness, and distributions of each table in `cms_data.db`.

Run after `01_setup.ipynb` has created the database.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
DB = Path("../cms_data.db")
conn = sqlite3.connect(DB)

In [ ]:
def inspect(table_name: str) -> None:
    """Load table from DB and inspect its structure and content."""
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn)

    print(f"{'='*70}")
    print(f"  {table_name}")
    print(f"  {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
    print(f"{'='*70}")

    print("\n--- dtypes ---")
    print(df.dtypes.to_string())

    print("\n--- sample (3 rows) ---")
    display(df.head(3))

    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if not nulls.empty:
        print("\n--- null counts (non-zero columns only) ---")
        print(nulls.to_string())

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    cost_cols = [c for c in numeric_cols if any(k in c for k in ("AMT", "CST", "PAY", "REIMB", "RES"))]
    if cost_cols:
        print("\n--- cost / payment columns ---")
        display(df[cost_cols].describe().round(2))

---
## Inpatient Claims

In [ ]:
inspect("inpatient_claims")

---
## Outpatient Claims

In [ ]:
inspect("outpatient_claims")

---
## Prescription Drug Events

In [ ]:
inspect("prescription_drug_events")

---
## Beneficiary Summary

In [ ]:
inspect("beneficiary_summary")

---
## Chronic Condition Prevalence

In [ ]:
bene = pd.read_sql("SELECT * FROM beneficiary_summary", conn)

sp_cols = [c for c in bene.columns if c.startswith("SP_")]
prevalence = (bene[sp_cols] == 1).mean().mul(100).round(1).sort_values(ascending=False)

print("Chronic condition prevalence (% of beneficiary-years with condition = 1):\n")
print(prevalence.to_string())

In [ ]:
conn.close()